# Geoloc

## infos
ligne 42368 : photo id 5464485473, correction -> les dates étaient décalée, le # de minutes (25) était collé au titre de la photo ("une lundi matin comme tout les autre ;-(") et le décalage était propagé.

pour les autres, on enleve les 142 lignes qui ont des valeurs non-nulles qui dépassent les colonnes attendues. (unnamed 16, y'a aussi 2 lignes avec des valeurs dans unnamed 18 mais elles sont comptabilisées dans les autres)

pour data cleaning : tuple identiques excepté les tags, un user avec plein de photos au même point gps (carousel), 

In [1]:
import numpy as np
import pandas as pd
import folium as fl

In [2]:
read_data = pd.read_csv("./flickr_data2.csv")
len(read_data)

C:\Users\celie\AppData\Local\Temp\ipykernel_15800\1284480133.py:1: DtypeWarning: Columns (11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  read_data = pd.read_csv("./flickr_data2.csv")


420240

In [12]:
wrong_data = read_data.dropna(how="all", subset=read_data.columns[[16, 18]])
fix_data = read_data.drop(index=wrong_data.index, axis=1)

toDropColumns = read_data.columns[[16, 17, 18]]
fix_data = fix_data.drop(toDropColumns, axis=1)

len(fix_data)

420098

Retirer les tuples dupliquée (ignore id et tags pour enlever certains dupliqué quand un post est modifié et certains carousels, passe de 187544 à 175688 donc pas tant que ça, la majorité sont dupliqués point barre)

In [13]:
fix_data.drop_duplicates(inplace=True, subset=fix_data.columns.drop(["id", " tags"]), keep='last')
len(fix_data)

175688

Retirer les carousels

In [15]:
fix_data.columns

Index(['id', ' user', ' lat', ' long', ' tags', ' title', ' date_taken_minute',
       ' date_taken_hour', ' date_taken_day', ' date_taken_month',
       ' date_taken_year', ' date_upload_minute', ' date_upload_hour',
       ' date_upload_day', ' date_upload_month', ' date_upload_year'],
      dtype='object')

In [14]:
fix_data.drop_duplicates(inplace=True, subset=fix_data.columns.drop(["id", " title"]))
len(fix_data)

161122

Retirer si tags et titre sont null

In [16]:
text_data = fix_data.dropna(how='all', subset=[" tags", " title"])
len(text_data)

151458

## Partie sur la carte.

In [ ]:
map = fl.Map(location=(45.757778, 4.832222), zoom_start = 13)

In [29]:
fl.Marker(
    location=[45.757778, 4.832222],
    tooltip="Clique moi !",
    popup="Centre de Lyon<br>(selon wikipedia)",
    icon=fl.Icon(icon_color="white", icon="info-sign", color="red"),
).add_to(map)

fl.Marker(
    location=[45.781883, 4.872814],
    tooltip="Clique moi !",
    popup="Ada Lovelace<br>(notre bâtiment)",
    icon=fl.Icon(icon_color="white", icon="book", color="blue"),
).add_to(map)

In [ ]:
trail_coordinates = [
    (45.757778, 4.832222),
    (45.765482, 4.852781),
    (45.781883, 4.872814),
]

fl.PolyLine(trail_coordinates, tooltip="Coast", color="green").add_to(map)
fl.Polygon()

In [82]:
map.save("test.html")

# Text mining

In [ ]:
def sepTags(str):
    tags = []
    i = 0
    j = 0
    while i < len(str):
        c = str[i]
        if i == len(str)-1:
            tags.append(str[j:i+1])
        if c == ",":
            tags.append(str[j:i])
            j = i+1
        i = i+1
    return tags
sepTags("str, stts,rg")

['str', ' stts', 'rg']

In [46]:
tags_data = fix_data[" tags"]
tags = []
for i, t in tags_data.items():
    if pd.isna(t):
        val = []
    else:
        val = sepTags(t)
    tags.append(val)

In [50]:
firstTag = tags[0][0]
count = 0
for t in tags:
    for a in t:
        if a == firstTag:
            count += 1
print(firstTag, " : ", count)
# puis 


lyon  :  62951
